## Reading JSON files from S3 bucket and converting into delta table and saving into Bronze schema Databricks unity catalog.
#### Booking data conversion.

In [0]:
raw_bookings_path = (
    "s3://voyage-data-plateform-yuvaraj-india/"
    "raw/airbyte/voyage_raw/bookings/"
)

df_bookings = spark.read.json(raw_bookings_path)

print(f"Raw booking records read: {df_bookings.count()}")

display(df_bookings.limit(10))

In [0]:
df_bookings.printSchema()

In [0]:
bronze_bookings = df_bookings.select(
    "_airbyte_data.*",
    "_airbyte_extracted_at",
    "_airbyte_generation_id",
    "_airbyte_meta",
    "_airbyte_raw_id",
)

bronze_bookings.printSchema()

In [0]:
print(f"Bronze rows: {bronze_bookings.count()}")
print(f"Distinct booking IDs: {bronze_bookings.select('booking_id').distinct().count()}")

display(bronze_bookings.limit(10))

Inspection of Duplicates.

In [0]:
from pyspark.sql import functions as F

duplicate_bookings = (
    bronze_bookings
    .groupBy("booking_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

print(f"Booking IDs with multiple versions: {duplicate_bookings.count()}")

display(duplicate_bookings.limit(20))

In [0]:
display(
    bronze_bookings
    .filter(F.col("booking_id").isin(
        [row["booking_id"] for row in duplicate_bookings.limit(10).collect()]
    ))
    .orderBy("booking_id", "updated_at")
)

In [0]:
(
    bronze_bookings.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("voyage.bronze.bookings")
)

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT booking_id) AS unique_booking_ids,
    COUNT(*) - COUNT(DISTINCT booking_id) AS duplicate_versions
FROM voyage.bronze.bookings;

In [0]:
%sql
SELECT
    booking_id,
    updated_at,
    _airbyte_extracted_at,
    _airbyte_generation_id,
    _airbyte_raw_id
FROM voyage.bronze.bookings
WHERE booking_id IN ('BK000001', 'BK000003')
ORDER BY booking_id, updated_at;

### Customer delta table.

In [0]:
raw_customers_path = (
    "s3://voyage-data-plateform-yuvaraj-india/"
    "raw/airbyte/voyage_raw/customers/"
)

df_customers = spark.read.json(raw_customers_path)

print(f"Raw customer records read: {df_customers.count()}")

display(df_customers.limit(10))

In [0]:
df_customers.printSchema()

In [0]:
bronze_customers = df_customers.select(
    "_airbyte_data.*",
    "_airbyte_extracted_at",
    "_airbyte_generation_id",
    "_airbyte_meta",
    "_airbyte_raw_id",
)

bronze_customers.printSchema()

In [0]:
print(f"Bronze rows: {bronze_customers.count()}")
print(f"Distinct booking IDs: {bronze_customers.select('customer_id').distinct().count()}")

display(bronze_customers.limit(10))

In [0]:
from pyspark.sql import functions as F

duplicate_customers = (
    bronze_customers
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

print(f"Customer IDs with multiple versions: {duplicate_customers.count()}")

display(duplicate_customers.limit(20))

In [0]:
display(
    bronze_customers
    .filter(F.col("customer_id").isin(
        [row["customer_id"] for row in duplicate_customers.limit(10).collect()]
    ))
    .orderBy("customer_id", "updated_at")
)

In [0]:
(
    bronze_customers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("voyage.bronze.customers")
)

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS unique_booking_ids,
    COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_versions
FROM voyage.bronze.customers;

In [0]:
%sql
SELECT
    customer_id,
    updated_at,
    _airbyte_extracted_at,
    _airbyte_generation_id,
    _airbyte_raw_id
FROM voyage.bronze.customers
WHERE customer_id IN ('CUS073715', 'CUS073716')
ORDER BY customer_id, updated_at;

#### Tenants and Properties delta Table.

In [0]:
raw_tenants_path = (
    "s3://voyage-data-plateform-yuvaraj-india/"
    "raw/airbyte/voyage_raw/tenants/"
)

df_tenants = spark.read.json(raw_tenants_path)

print(f"Raw Tenant records: {df_tenants.count()}")

display(df_tenants.limit(10))
df_tenants.printSchema()

In [0]:
bronze_tenants = df_tenants.select(
    "_airbyte_data.*",
    "_airbyte_extracted_at",
    "_airbyte_generation_id",
    "_airbyte_meta",
    "_airbyte_raw_id",
)

bronze_tenants.printSchema()
print(f"Bronze rows: {bronze_tenants.count()}")
print(f"Distinct booking IDs: {bronze_tenants.select('tenant_id').distinct().count()}")

display(bronze_tenants.limit(10))

from pyspark.sql import functions as F

duplicate_tenants = (
    bronze_tenants
    .groupBy("tenant_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

print(f"Tenant IDs with multiple versions: {duplicate_tenants.count()}")

display(duplicate_tenants.limit(20))
display(
    bronze_tenants
    .filter(F.col("tenant_id").isin(
        [row["tenant_id"] for row in duplicate_tenants.limit(10).collect()]
    ))
    .orderBy("tenant_id", "updated_at")
)

In [0]:
(
    bronze_tenants.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("voyage.bronze.tenants")
)

In [0]:
%sql
SELECT COUNT(*) AS Total_Tenants
FROM voyage.bronze.tenants;

#### Properties Delta Table using S3 JSON files.

In [0]:
raw_properties_path=(
    "s3://voyage-data-plateform-yuvaraj-india/"
    "raw/airbyte/voyage_raw/properties/"
)
df_properties=spark.read.json(raw_properties_path)
print(f"Raw property records: {df_properties.count()}")
display(df_properties.limit(10))
df_properties.printSchema()

In [0]:
bronze_properties = df_properties.select(
    "_airbyte_data.*",
    "_airbyte_extracted_at",
    "_airbyte_generation_id",
    "_airbyte_meta",
    "_airbyte_raw_id",
)

bronze_properties.printSchema()
print(f"Bronze rows: {bronze_properties.count()}")
print(f"Distinct booking IDs: {bronze_properties.select('property_id').distinct().count()}")

display(bronze_properties.limit(10))

from pyspark.sql import functions as F

duplicate_properties = (
    bronze_properties
    .groupBy("property_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

print(f"Property IDs with multiple versions: {duplicate_properties.count()}")

display(duplicate_properties.limit(20))
display(
    bronze_properties
    .filter(F.col("property_id").isin(
        [row["property_id"] for row in duplicate_properties.limit(10).collect()]
    ))
    .orderBy("property_id", "updated_at")
)

In [0]:
(
    bronze_properties.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("voyage.bronze.properties")
)

#### Validation Quary for all the Delta tables.

In [0]:
%sql
SELECT 'tenants' AS table_name, COUNT(*) AS rows
FROM voyage.bronze.tenants

UNION ALL

SELECT 'properties', COUNT(*)
FROM voyage.bronze.properties

UNION ALL

SELECT 'customers', COUNT(*)
FROM voyage.bronze.customers

UNION ALL

SELECT 'bookings', COUNT(*)
FROM voyage.bronze.bookings;